# 03 — Well-tempered metadynamics

**Learning goals**

- add a history-dependent bias along a collective variable (here simply $x$) and let it act on the dynamics,
- understand well-tempered hill attenuation and the bias factor $\gamma_\mathrm{WT}$,
- reconstruct $F(x)$ from the bias and judge how converged it is,
- replay the run as an animation (simulation first, animation afterwards).

In [ ]:
# Setup: make the `src` package importable (locally or on Google Colab).
import sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/HyeonjeYang/md-enhanced-sampling-lab.git"
if "google.colab" in sys.modules:
    ROOT = Path("/content/md-enhanced-sampling-lab")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
else:
    ROOT = Path.cwd().resolve()
    if not (ROOT / "src").is_dir():
        ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import time as timer
import inspect
import numpy as np
import matplotlib.pyplot as plt

from src.potentials import double_well_potential, double_well_force, double_well_landmarks, true_free_energy
from src.langevin import brownian_dynamics
from src.metadynamics import (run_wt_metadynamics, metad_free_energy, gaussian_hill, wt_hill_height,
                              delta_T_from_bias_factor)
from src.analysis import count_transitions, free_energy_rmse
from src.plotting import (set_style, save_figure, plot_reference, free_energy_axes, animate_metadynamics,
                          show_animation, PALETTE, REFERENCE, METHOD_COLORS)
set_style()

## Parameters

The four metadynamics parameters worth experimenting with are `hill_height`, `hill_sigma`,
`deposition_stride` and `bias_factor`.

In [ ]:
a, b = 2.0, 0.5
T = 0.4                         # kT (k_B = 1)
friction = 1.0
dt = 0.002
n_steps = 200_000               # 400 time units

hill_height = 0.1               # initial Gaussian height w0 (= 0.25 kT)
hill_sigma = 0.1                # Gaussian width along x
deposition_stride = 250         # steps between hills (0.5 time units)
bias_factor = 8.0               # gamma_WT = (T + Delta T) / T

x_grid = np.linspace(-2.5, 2.5, 501)   # bias grid; must cover everything the walker visits
snapshot_stride = 4000          # store the bias every 4000 steps (50 animation frames)
seed = 2026

## 1. Well-tempered metadynamics in a nutshell

The walker evolves on an effective potential that changes with time,

$$V_\mathrm{eff}(x,t) = V(x) + V_\mathrm{bias}(x,t), \qquad
F_\mathrm{total} = -\frac{\partial V}{\partial x} - \frac{\partial V_\mathrm{bias}}{\partial x},$$

where the bias is a sum of Gaussians deposited at the visited positions $s_i$:

$$V_\mathrm{bias}(s,t) = \sum_i w_i \exp\!\left[-\frac{(s-s_i)^2}{2\sigma^2}\right], \qquad
w(t) = w_0\exp\!\left[-\frac{V_\mathrm{bias}(s(t),t)}{k_B\Delta T}\right].$$

Hills become smaller where bias has already accumulated. With $\gamma_\mathrm{WT} = (T+\Delta T)/T$ the bias
approaches $V_\mathrm{bias} \to -(1-1/\gamma_\mathrm{WT})F + C$ at long times, so

$$F_\mathrm{MetaD}(s) = -\frac{T+\Delta T}{\Delta T}V_\mathrm{bias}(s) = -\frac{\gamma_\mathrm{WT}}{\gamma_\mathrm{WT}-1}V_\mathrm{bias}(s) + C.$$

In [ ]:
lm = double_well_landmarks(a, b)
kB_dT = delta_T_from_bias_factor(T, bias_factor)
print(f"Delta T = {kB_dT:.2f}  (T + Delta T = {T + kB_dT:.2f}),  barrier from the left well = {lm['barrier_from_left'] / T:.1f} kT")
print(f"long-time effective barrier ~ barrier / gamma = {lm['barrier_from_left'] / bias_factor / T:.1f} kT")

## 2. Grid-based implementation

Each deposition adds one Gaussian and its analytical derivative to arrays on a fixed grid; each step only
interpolates the stored derivative. The cost per step does not grow with the number of hills. The core of the loop
in `src/metadynamics.py`:

```python
f_bias = -np.interp(x, x_grid, dbias)                      # bias force at the current position
x += (force(x) + f_bias) / friction * dt + noise[step - 1]  # the bias acts on the dynamics
if step % deposition_stride == 0:
    w = wt_hill_height(np.interp(x, x_grid, bias), hill_height, kB_delta_T)
    g, dg = gaussian_hill(x_grid, x, w, hill_sigma)         # vectorized over the grid
    bias += g
    dbias += dg
```

In [ ]:
print(inspect.getsource(gaussian_hill))
print(inspect.getsource(wt_hill_height))

## 3. Run WT-MetaD and an unbiased reference of the same length

In [ ]:
force = lambda x: double_well_force(x, a, b)

t0 = timer.perf_counter()
metad = run_wt_metadynamics(lm["x_left"], force, n_steps, dt, T, np.random.default_rng(seed),
                            hill_height=hill_height, hill_sigma=hill_sigma,
                            deposition_stride=deposition_stride, bias_factor=bias_factor,
                            x_grid=x_grid, friction=friction, traj_stride=10,
                            snapshot_stride=snapshot_stride)
print(f"WT-MetaD: {timer.perf_counter() - t0:.1f} s, {len(metad['hill_center'])} hills, "
      f"first/last hill height {metad['hill_height'][0]:.3f} / {metad['hill_height'][-1]:.4f}")

unbiased = brownian_dynamics(lm["x_left"], force, n_steps, dt, T, np.random.default_rng(seed), friction, stride=10)

core = (lm["x_left"] + 0.3, lm["x_right"] - 0.3)
print(f"transitions: unbiased {count_transitions(unbiased, *core)},  WT-MetaD {count_transitions(metad['x'], *core)}")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 4.2), sharex=True, sharey=True)
axes[0].plot(metad["time"], unbiased, lw=0.5, color=METHOD_COLORS["unbiased"])
axes[0].set_title("unbiased Brownian dynamics", fontsize=10)
axes[1].plot(metad["time"], metad["x"], lw=0.5, color=METHOD_COLORS["metad"])
axes[1].set_title("well-tempered metadynamics", fontsize=10)
for ax in axes:
    ax.axhline(lm["x_barrier"], color=REFERENCE, ls=":", lw=1)
    ax.set_ylabel(r"$x(t)$")
axes[1].set_xlabel("time (reduced units)")
fig.tight_layout()
plt.show()

In this finite toy simulation the bias fills the starting well within a few tens of time units, after which the
walker crosses the barrier repeatedly.

## 4. Hill attenuation and the effective potential

In [ ]:
V_grid = double_well_potential(x_grid, a, b)
F_grid = true_free_energy(x_grid, a, b)
V_eff = V_grid + metad["bias"]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
axes[0].plot(metad["hill_time"], metad["hill_height"], ".", ms=3, color=METHOD_COLORS["metad"])
axes[0].set_xlabel("time")
axes[0].set_ylabel(r"hill height $w(t)$")
axes[0].set_yscale("log")
axes[1].plot(x_grid, F_grid, color=REFERENCE, lw=1.2, label=r"$V(x) - \min V$")
axes[1].plot(x_grid, metad["bias"], color=METHOD_COLORS["metad"], label=r"$V_\mathrm{bias}$ (final)")
axes[1].plot(x_grid, V_eff - V_eff[F_grid < 3.3].min(), color=PALETTE[0], label=r"$V + V_\mathrm{bias}$ (shifted)")
axes[1].plot(x_grid, F_grid / bias_factor, ls=":", color=PALETTE[0], label=r"$F/\gamma_\mathrm{WT}$ (long-time limit)")
axes[1].set_xlim(-1.8, 1.8)
axes[1].set_ylim(-0.2, 4.5)
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel("energy")
axes[1].legend(fontsize=8)
fig.tight_layout()
plt.show()

The hill height decays as the bias grows (the well-tempered rule). The effective potential is not flat: in the
long-time limit it approaches $F/\gamma_\mathrm{WT}$, a smoothed version of the original landscape, which is why
the walker samples the barrier region but does not wander off to arbitrarily high energies.

## 5. Free-energy reconstruction

Both profiles are shifted so that their minimum is zero. The comparison region is where
$F_\mathrm{true} \le F_\mathrm{true}(\text{barrier}) + 2k_BT$, i.e. both basins and the barrier.

In [ ]:
region = F_grid <= lm["barrier_from_left"] + 2 * T
F_metad = metad_free_energy(metad["bias"], bias_factor)
rmse, coverage = free_energy_rmse(F_metad, F_grid, region)
print(f"RMSE over the comparison region: {rmse:.3f} = {rmse / T:.2f} kT (coverage {coverage:.0%})")

fig, ax = plt.subplots(figsize=(6.5, 3.6))
plot_reference(ax, x_grid, F_grid)
for i, alpha in zip([5, 15, len(metad["snapshot_bias"]) - 1], [0.35, 0.6, 1.0]):
    F_i = metad_free_energy(metad["snapshot_bias"][i], bias_factor)
    ax.plot(x_grid, F_i, color=METHOD_COLORS["metad"], alpha=alpha,
            label=rf"$F_\mathrm{{MetaD}}$, t = {metad['snapshot_time'][i]:.0f}")
ax.set_xlim(-1.8, 1.8)
free_energy_axes(ax, ylim=(-0.2, 4.5))
ax.legend(fontsize=8)
fig.tight_layout()
save_figure(fig, "03_metad_reconstruction")
plt.show()

## 6. Is it converged?

Track the error of the instantaneous estimate over time. A finite run never gives an exactly converged bias;
the curve flattens but keeps fluctuating because new hills are still being added.

In [ ]:
rmse_t = [free_energy_rmse(metad_free_energy(bias_i, bias_factor), F_grid, region)[0]
          for bias_i in metad["snapshot_bias"]]
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(metad["snapshot_time"], np.array(rmse_t) / T, "o-", ms=3, color=METHOD_COLORS["metad"])
ax.set_xlabel("time")
ax.set_ylabel(r"RMSE / $k_BT$")
ax.set_yscale("log")
fig.tight_layout()
plt.show()

## 7. Animation

The simulation above already stored 50 sparse snapshots of the bias; the animation only replays them.
It is embedded as HTML/JavaScript, so it needs no ffmpeg.

In [ ]:
MAKE_ANIMATION = True
if MAKE_ANIMATION:
    anim = animate_metadynamics(metad, V_grid, F_grid, bias_factor, ylim=(-0.2, 5.0), xlim=(-1.8, 1.8))
    display(show_animation(anim))

## 8. Optional parameter sweep

Small sweep over `hill_sigma` and `bias_factor` (about 1–2 s per run). Things to look for: very wide hills smear
out the barrier; a small bias factor keeps the effective barrier high and slows exploration; a very large one
behaves more like standard (non-tempered) metadynamics and converges more noisily.

In [ ]:
RUN_SWEEP = False
if RUN_SWEEP:
    print(f"{'sigma':>6} {'gamma':>6} {'transitions':>12} {'RMSE/kT':>8}")
    for sig in (0.05, 0.1, 0.3):
        for gam in (3.0, 8.0, 30.0):
            r = run_wt_metadynamics(lm["x_left"], force, n_steps, dt, T, np.random.default_rng(seed),
                                    hill_height=hill_height, hill_sigma=sig, deposition_stride=deposition_stride,
                                    bias_factor=gam, x_grid=x_grid, friction=friction)
            err = free_energy_rmse(metad_free_energy(r["bias"], gam), F_grid, region)[0]
            print(f"{sig:6.2f} {gam:6.1f} {count_transitions(r['x'], *core):12d} {err / T:8.2f}")

## Takeaways

- WT-MetaD deposits Gaussians along a chosen CV; their gradient is an extra force that pushes the walker out of visited regions.
- Hill heights decay as $\exp(-V_\mathrm{bias}/k_B\Delta T)$, and the bias converges towards $-(1-1/\gamma_\mathrm{WT})F$.
- $F_\mathrm{MetaD} = -\frac{\gamma_\mathrm{WT}}{\gamma_\mathrm{WT}-1}V_\mathrm{bias}$, compared only after aligning additive constants and only where bias was deposited.
- In this 1D toy the reconstructed profile approaches the known reference within a fraction of $k_BT$; a finite run is never exactly converged, and a poor CV would break the method.

*References:* Laio & Parrinello, *PNAS* **99**, 12562 (2002), doi:10.1073/pnas.202427399 ·
Barducci, Bussi & Parrinello, *Phys. Rev. Lett.* **100**, 020603 (2008), doi:10.1103/PhysRevLett.100.020603.